DH演習2026 · 教材 第38版 · 2026-r38 · 2026-10-15

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第38版
教材ID: 2026-r38
教材日付: 2026-10-15
活動: source_trial_illustrated
環境: Colab / Jupyter
ソース: 0168ecfac6edcdfec02b46504654b08d76a08d2f
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第2回：くずし字のOCR結果を画像と照合する

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料：[画本虫撰](https://codh.rois.ac.jp/pmjt/book/200014778/)（日本古典籍データセット、国文研所蔵、CODH配信、[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)）

## 1. 画像を取得する

`region`は切り出す範囲（左,上,幅,高さ）で、原画像のピクセルで指定します。見開き全体は`"full"`です。

In [ ]:
#@title 操作：道具を読み込む
from PIL import Image

In [ ]:
koma = 5
region = "842,365,3875,3061"
url = f"https://codh.rois.ac.jp/pmjt/iiif/200014778/200014778_{koma:05d}.tif/{region}/2400,/0/default.jpg"
!curl -fso page.jpg {url}
Image.open("page.jpg")

## 2. 文字を認識する

ノートブックは[NDL古典籍OCR-Lite](https://github.com/ndl-lab/ndlkotenocr-lite/tree/ede4283845cdc0ba2bda8b7ebfc3dc80b33c92c8)を取得し、`page.jpg`を認識した結果を`text`に入れます。初回だけ、準備に約20秒かかります。

In [ ]:
#@title 操作：OCRの実行環境を準備する
ocr = "https://github.com/ndl-lab/ndlkotenocr-lite/archive/ede4283845cdc0ba2bda8b7ebfc3dc80b33c92c8.tar.gz"
!mkdir -p koten && curl -fsL {ocr} | tar xz -C koten --strip-components 1

In [ ]:
!cd koten/src && uv run -q --with-requirements ../requirements.txt python ocr.py --sourceimg ../../page.jpg --output ../.. --viz True

text = open("page.txt").read()
print(text)

## 3. 画像と照合する

枠はOCRが認識した行です。見出しの「馬追虫」「むかて」と、署名の「唐衣橘洲」を画像と比べてください。

In [ ]:
Image.open("viz_page.jpg")

## 4. 濁点を補って検索する

画像の見出しは「むかて」で、現代の読みは「むかで」です。濁点を補うのは誤認識の訂正ではなく表記の正規化なので、OCR結果の`text`を残し、別の変数に入れます。実行する前に、補った後の出現回数を予測してください。

In [ ]:
query = "むかで"
normalized = text.replace("むかて", "むかで")
print("補う前:", text.count(query))
print("補った後:", normalized.count(query))

数えているのは文字列の一致です。虫の数や詩の数ではありません。

## 5. 別の資料で試す

IIIFに対応した資料なら、1.の`url`をその資料の画像URLに変えると、同じ手順で認識できます。

## 確認問題

見出しの虫の名前を、現代の読みのひらがなで検索できるようにします。`readings`に「OCR結果の表記: 読み」を書き足し、`normalized`に「うまおいむし」と「むかで」が両方含まれるようにしてください。`text`は変えないでください。

In [ ]:
readings = {"むかて": "むかで"}

normalized = text
for written, reading in readings.items():
    normalized = normalized.replace(written, reading)
print(normalized)

In [ ]:
#@title 答えを確かめる
missing = [word for word in ("うまおいむし", "むかで") if word not in normalized]
if "馬追虫" not in text:
    print("結果：不合格（`text`が変わっています。2.から実行し直してください）")
elif missing:
    print(f"結果：不合格（{'・'.join(missing)}が見つかりません）")
else:
    print("結果：合格")